In [44]:
from typing import List, Dict, Any, Optional
import numpy as np
from PIL import Image
import torch
from transformers import pipeline
from dataclasses import dataclass

@dataclass
class BoundingBox:
    xmin: int
    ymin: int
    xmax: int
    ymax: int

    @property
    def xyxy(self) -> List[float]:
        return [self.xmin, self.ymin, self.xmax, self.ymax]
    
class DetectionResult:
    score: float
    label: str
    box: BoundingBox
    mask: Optional[np.array] = None

    @classmethod
    def from_dict(cls, detection_dict: Dict) -> 'DetectionResult':
        return cls(score=detection_dict['score'],
                   label=detection_dict['label'],
                   box=BoundingBox(xmin=detection_dict['box']['xmin'],
                                   ymin=detection_dict['box']['ymin'],
                                   xmax=detection_dict['box']['xmax'],
                                   ymax=detection_dict['box']['ymax']))
        
def detect(
    image: Image.Image,
    labels: List[str],
    threshold: float = 0.3,
    detector_id: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Use Grounding DINO to detect a set of labels in an image in a zero-shot fashion.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    detector_id = detector_id if detector_id is not None else "IDEA-Research/grounding-dino-tiny"
    object_detector = pipeline(model=detector_id, task="zero-shot-object-detection", device=device)

    labels = [label if label.endswith(".") else label+"." for label in labels]

    results = object_detector(image,  candidate_labels=labels, threshold=threshold)
    results = [DetectionResult.from_dict(result) for result in results]

    return results

In [45]:
import pickle
import os

path = '/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_llama3_sg_grounding_temp_1_topp_1_dp_1000.pkl_1159364.pkl'

with open(path, 'rb') as f:
    data = pickle.load(f)

In [48]:
tuples = data[0]['response_triples']
labels = ""
for t in tuples:
    labels += t[0] + "."
    
image_path = os.path.join('/home/ubuntu/Multimodal-Uncertainty-Quantification/dataset/GQA/images', '1159364' + '.jpg')

In [49]:
threshold = 0.3
detector_id = None
image = Image.open(image_path)
detections = detect(image, labels, threshold, detector_id)

TypeError: DetectionResult() takes no arguments